# Western Visayas Flood Prediction — NB2: Model Training & SHAP

Trains and compares four classifiers on the spatial-temporal dataset from NB1, under a
**time-aware** evaluation protocol, with **algorithm-level class-imbalance handling**
(class weighting / scale_pos_weight — no SMOTE) and **SHAP** explainability.

- **Label:** `flood_observed` (real DSWD DROMIC flood days), not a rainfall threshold.
- **Features:** rainfall + antecedent rainfall + typhoon + terrain (elevation, slope, TWI, land cover) + calendar.
- **Primary metric:** recall (a missed flood is costlier than a false alarm), with ROC-AUC, precision, F1.

Author: Claude L. Calimlim - Independent Researcher


## 1. Setup

In [ ]:
!pip install xgboost shap scikit-learn --quiet

import warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import shap
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix, roc_curve)
from sklearn.model_selection import StratifiedKFold
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

BASE    = Path('/content/drive/MyDrive/wv_flood_research')
OUT_DIR = BASE
RANDOM_STATE = 42


## 2. Load dataset & define features

Seven dynamic + terrain features. `landcover` is categorical (one-hot). `province`, `lat`, `lon`,
`date`, `pt_idx` are identifiers, not model inputs.

In [ ]:
df = pd.read_parquet(OUT_DIR / 'wv_flood_dataset_v2.parquet')
df['date'] = pd.to_datetime(df['date'])
print('Dataset:', df.shape, '| years', df.date.dt.year.min(), '-', df.date.dt.year.max())
print('Positive rate:', round(100*df.flood_observed.mean(), 3), '%')

# season ordinal encode
season_map = {'amihan': 0, 'transition': 1, 'habagat': 2}
df['season_enc'] = df['season'].map(season_map)

# one-hot land cover (static terrain category)
lc_dummies = pd.get_dummies(df['landcover'].astype('Int64'), prefix='lc')
df = pd.concat([df, lc_dummies], axis=1)

FEATURES = (['rainfall_mm', 'rain_3day_mm', 'rain_7day_mm', 'consec_rain_days',
             'month', 'season_enc', 'typhoon_active',
             'elevation', 'slope', 'TWI'] + list(lc_dummies.columns))
TARGET = 'flood_observed'
print(f'{len(FEATURES)} features:', FEATURES)


## 3. Time-aware train/test split

Chronological split prevents temporal leakage. Train on earlier years, test on the most recent.
(With the 2014-2024 window, we train 2014-2021 and test 2022-2024 — adjust as needed.)

In [ ]:
TRAIN_END = 2021   # inclusive; test is TRAIN_END+1 .. end
train = df[df.date.dt.year <= TRAIN_END].copy()
test  = df[df.date.dt.year >  TRAIN_END].copy()

X_train, y_train = train[FEATURES].fillna(0), train[TARGET]
X_test,  y_test  = test[FEATURES].fillna(0),  test[TARGET]
print(f'Train: {X_train.shape} | positives {y_train.sum()}')
print(f'Test:  {X_test.shape} | positives {y_test.sum()}')

# scale for LR + SVM only (tree models are scale-invariant)
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)

neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
scale_pos_weight = neg / max(pos, 1)
print('scale_pos_weight:', round(scale_pos_weight, 2))


## 4. Train the four models

Class imbalance handled at the algorithm level: `class_weight='balanced'` for LR / RF / SVM,
`scale_pos_weight` for XGBoost. No resampling (SMOTE) — preserves the physical distribution.

In [ ]:
models = {}

models['Logistic Regression'] = LogisticRegression(
    class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)
models['Random Forest'] = RandomForestClassifier(
    n_estimators=200, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
models['XGBoost'] = XGBClassifier(
    n_estimators=200, scale_pos_weight=scale_pos_weight, random_state=RANDOM_STATE,
    eval_metric='logloss', n_jobs=-1)
models['SVM'] = SVC(
    kernel='rbf', class_weight='balanced', probability=True, random_state=RANDOM_STATE)

# LR & SVM use scaled features; RF & XGB use raw
uses_scaled = {'Logistic Regression', 'SVM'}
for name, m in models.items():
    Xtr = X_train_s if name in uses_scaled else X_train.values
    m.fit(Xtr, y_train)
    print(f'trained {name}')


## 5. Evaluate on the held-out test set

In [ ]:
rows = []
proba = {}
for name, m in models.items():
    Xte = X_test_s if name in uses_scaled else X_test.values
    p = m.predict(Xte)
    pr = m.predict_proba(Xte)[:, 1]
    proba[name] = pr
    rows.append({
        'Model': name,
        'Accuracy':  round(accuracy_score(y_test, p), 4),
        'Precision': round(precision_score(y_test, p, zero_division=0), 4),
        'Recall':    round(recall_score(y_test, p, zero_division=0), 4),
        'F1':        round(f1_score(y_test, p, zero_division=0), 4),
        'ROC-AUC':   round(roc_auc_score(y_test, pr), 4),
    })
results = pd.DataFrame(rows).sort_values('ROC-AUC', ascending=False).reset_index(drop=True)
print(results.to_string(index=False))
results.to_csv(OUT_DIR / 'model_results_v2.csv', index=False)


## 6. Confusion matrices (false negatives = missed floods)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for ax, name in zip(axes.ravel(), models):
    Xte = X_test_s if name in uses_scaled else X_test.values
    cm = confusion_matrix(y_test, models[name].predict(Xte))
    im = ax.imshow(cm, cmap='Blues')
    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, str(v), ha='center', va='center',
                color='white' if v > cm.max()/2 else 'black', fontsize=13)
    rec = recall_score(y_test, models[name].predict(Xte), zero_division=0)
    auc = roc_auc_score(y_test, proba[name])
    ax.set_title(f'{name}\nRecall={rec:.3f} | ROC-AUC={auc:.3f}')
    ax.set_xticks([0, 1]); ax.set_xticklabels(['No Flood', 'Flood'])
    ax.set_yticks([0, 1]); ax.set_yticklabels(['No Flood', 'Flood'])
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.suptitle('Confusion Matrices - Test Set', fontweight='bold')
plt.tight_layout(); plt.savefig(OUT_DIR / 'confusion_matrices_v2.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. ROC curves

In [ ]:
plt.figure(figsize=(8, 7))
for name in models:
    fpr, tpr, _ = roc_curve(y_test, proba[name])
    plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {roc_auc_score(y_test, proba[name]):.4f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random (AUC = 0.5)')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curves - Test Set', fontweight='bold')
plt.legend(loc='lower right'); plt.grid(alpha=0.3)
plt.savefig(OUT_DIR / 'roc_curves_v2.png', dpi=150, bbox_inches='tight')
plt.show()


## 8. Cross-validation (stratified 5-fold on training set)

Checks stability beyond the single chronological split.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
for name, m in models.items():
    aucs, recs, f1s, precs = [], [], [], []
    Xcv = X_train_s if name in uses_scaled else X_train.values
    yv = y_train.values
    for tr, va in skf.split(Xcv, yv):
        from sklearn.base import clone
        c = clone(m); c.fit(Xcv[tr], yv[tr])
        pr = c.predict_proba(Xcv[va])[:, 1]; p = c.predict(Xcv[va])
        aucs.append(roc_auc_score(yv[va], pr))
        recs.append(recall_score(yv[va], p, zero_division=0))
        f1s.append(f1_score(yv[va], p, zero_division=0))
        precs.append(precision_score(yv[va], p, zero_division=0))
    cv_rows.append({
        'Model': name,
        'CV ROC-AUC':  f'{np.mean(aucs):.4f} ± {np.std(aucs):.4f}',
        'CV Recall':   f'{np.mean(recs):.4f} ± {np.std(recs):.4f}',
        'CV F1':       f'{np.mean(f1s):.4f} ± {np.std(f1s):.4f}',
        'CV Precision':f'{np.mean(precs):.4f} ± {np.std(precs):.4f}',
    })
cv = pd.DataFrame(cv_rows)
print(cv.to_string(index=False))
cv.to_csv(OUT_DIR / 'cv_results_v2.csv', index=False)


## 9. SHAP explainability

Explains the best-balanced model (Random Forest by default) on a test-set sample. Produces the
mean-|SHAP| ranking, beeswarm, and a dependence plot for the top feature.

In [ ]:
BEST = 'Random Forest'   # change to whichever you report as best-balanced
model = models[BEST]
Xexp = (X_test_s if BEST in uses_scaled else X_test.values)
Xexp_df = pd.DataFrame(Xexp, columns=FEATURES)

sample = Xexp_df.sample(min(2000, len(Xexp_df)), random_state=RANDOM_STATE)
explainer = shap.TreeExplainer(model)
sv = explainer.shap_values(sample)
sv1 = sv[1] if isinstance(sv, list) else sv   # class-1 contributions

# mean |SHAP| ranking
mean_abs = np.abs(sv1).mean(axis=0)
imp = (pd.DataFrame({'Feature': FEATURES, 'Mean |SHAP Value|': mean_abs})
       .sort_values('Mean |SHAP Value|', ascending=False).reset_index(drop=True))
print(imp.to_string(index=False))
imp.to_csv(OUT_DIR / 'shap_importance_v2.csv', index=False)

# beeswarm
shap.summary_plot(sv1, sample, show=False)
plt.tight_layout(); plt.savefig(OUT_DIR / 'shap_beeswarm_v2.png', dpi=150, bbox_inches='tight'); plt.show()

# bar
shap.summary_plot(sv1, sample, plot_type='bar', show=False)
plt.tight_layout(); plt.savefig(OUT_DIR / 'shap_bar_v2.png', dpi=150, bbox_inches='tight'); plt.show()

# dependence plot for the top feature
top = imp.iloc[0]['Feature']
shap.dependence_plot(top, sv1, sample, show=False)
plt.tight_layout(); plt.savefig(OUT_DIR / 'shap_dependence_v2.png', dpi=150, bbox_inches='tight'); plt.show()
